# 1. Raw

In [2]:
import numpy as np
import open3d as o3d

# Load data: assuming each point has 4 attributes (X, Y, Z, Intensity) as float32
raw_data = np.fromfile("0000000001.bin", dtype=np.float32)

# Reshape: -1 lets NumPy calculate the number of points; 4 is the attributes per point
points = raw_data.reshape(-1, 4)

# Extract only XYZ for visualization
xyz = points[:, :3]

# Convert to Open3D object for processing
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(xyz)

# o3d.visualization.draw_geometries([pcd])

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Intensity colourmap

In [3]:
intensity = points[:, 3]  # already in [0, 1] range for this dataset

# Convert to Open3D object for processing
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(xyz)

import matplotlib.cm as cm

# Apply a perceptually uniform colormap (viridis: purple→yellow)
colormap = cm.viridis(intensity)[:, :3]  # drop alpha channel
pcd.colors = o3d.utility.Vector3dVector(colormap)

# o3d.visualization.draw_geometries([pcd])

# 2. Cleaning

In [4]:
# Ego vehicle filter — remove LiDAR car's own body/mirrors
points_np = np.asarray(pcd.points)
ego_mask = ~(
    (np.abs(points_np[:, 0]) < 2.5) &   # ±2.5m along driving axis
    (np.abs(points_np[:, 1]) < 2)     # ±2m across (covers mirrors)
)
pcd = pcd.select_by_index(np.where(ego_mask)[0])
print(f"After ego removal: {len(pcd.points)} points")

After ego removal: 124767 points


In [5]:
# Discard far-range sparse returns and height outliers before further processing.
# Surveillance framing: keep ±40m footprint, Z in [-2.5, 1.5]m.
points_np = np.asarray(pcd.points)
mask = (
    (np.abs(points_np[:, 0]) < 40) &
    (np.abs(points_np[:, 1]) < 40) &
    (points_np[:, 2] > -2.5) &
    (points_np[:, 2] < 1.5)
)
pcd = pcd.select_by_index(np.where(mask)[0])
print(f"After ROI crop: {len(pcd.points)} points")

# Voxel downsampling
downpcd = pcd.voxel_down_sample(voxel_size=0.05)

# Statistical outlier removal
cl, ind = downpcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
cleaned_pcd = downpcd.select_by_index(ind)
print(f"After voxel + SOR: {len(cleaned_pcd.points)} points")

# Estimate normals (not required for DBSCAN, but useful later for analysis)
cleaned_pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30)
)

# o3d.visualization.draw_geometries([cleaned_pcd])

After ROI crop: 122456 points
After voxel + SOR: 85810 points


# 3. Ground removal

In [6]:
# Set global RNG seed (works for segment_plane, DBSCAN, etc.)
o3d.utility.random.seed(42)

# Primary ground: single-pass RANSAC (known good).
plane_model, inliers = cleaned_pcd.segment_plane(
    distance_threshold=0.2,
    ransac_n=3,
    num_iterations=1000,
    probability=1.0,  # forces all 1000 iterations; disables nondeterministic early termination
)
a, b, c, d = plane_model
print(f"Plane: {a:.2f}x + {b:.2f}y + {c:.2f}z + {d:.2f} = 0")

ground_cloud = cleaned_pcd.select_by_index(inliers)
remaining = cleaned_pcd.select_by_index(inliers, invert=True)

# Belt-and-braces band filter: reject points that sit just above the plane
# (residual ground — curbs, bumps, sidewalks 0.2-0.3m off the fitted plane
# that RANSAC's symmetric threshold missed). Uses the plane's actual normal,
# so it follows slopes correctly rather than a world-Z cut.
pts = np.asarray(remaining.points)
normal = np.array([a, b, c])
signed_dist = (pts @ normal + d) / np.linalg.norm(normal)

HEIGHT_ABOVE_PLANE = 0.3  # meters; below this = residual ground, reject
object_mask = signed_dist > HEIGHT_ABOVE_PLANE
objects_cloud = remaining.select_by_index(np.where(object_mask)[0])

print(f"Primary ground inliers: {len(ground_cloud.points)}, "
      f"residual stripped: {int((~object_mask).sum())}, "
      f"objects kept: {len(objects_cloud.points)}")

ground_cloud.paint_uniform_color([1.0, 0, 0])
# o3d.visualization.draw_geometries([ground_cloud, objects_cloud])

Plane: -0.02x + -0.02y + 1.00z + 1.70 = 0
Primary ground inliers: 37382, residual stripped: 2530, objects kept: 45898


PointCloud with 37382 points.

In [7]:
o3d.visualization.draw_geometries([ground_cloud])

In [8]:
o3d.visualization.draw_geometries([objects_cloud])

# 4. DBSCAN

In [9]:
import matplotlib.pyplot as plt

# Voxel size is 0.05m, so eps=0.3 ~ 6 voxel-widths. Enough gap to keep adjacent
# cars/pedestrians separated while still joining sparse distant returns.
with o3d.utility.VerbosityContextManager(o3d.utility.VerbosityLevel.Debug) as cm:
    labels = np.array(objects_cloud.cluster_dbscan(eps=0.3, min_points=10))

n_clusters = labels.max() + 1
n_noise = int((labels == -1).sum())
print(f"DBSCAN: {n_clusters} clusters, {n_noise} noise points "
      f"({n_noise / len(labels):.1%} noise)")

# Colour clusters (noise = black)
max_label = labels.max()
colors = plt.get_cmap("tab20")(labels / (max_label if max_label > 0 else 1))
colors[labels < 0] = 0
objects_cloud.colors = o3d.utility.Vector3dVector(colors[:, :3])

o3d.visualization.draw_geometries([objects_cloud])

[Open3D DEBUG] Precompute neighbors.
[Open3D DEBUG] Done Precompute neighbors.
[Open3D DEBUG] Compute Clusters
[Open3D DEBUG] Done Compute Clusters: 130
DBSCAN: 130 clusters, 1880 noise points (4.1% noise)


# 5. PointPillars DL Detection

KITTI-pretrained PointPillars running locally on CPU via `third_party/PointPillars`
(CUDA ops replaced with a CPU-compiled voxelizer + pure-torch NMS). Classes:
Car, Pedestrian, Cyclist.

In [10]:
import sys
from pathlib import Path
import torch

# Make the vendored repo importable
THIRD_PARTY = Path.cwd() / "third_party" / "PointPillars"
if str(THIRD_PARTY) not in sys.path:
    sys.path.insert(0, str(THIRD_PARTY))

from pointpillars.model import PointPillars

LABEL2CLASS = {0: "Pedestrian", 1: "Cyclist", 2: "Car"}
CLASS_COLOR = {
    "Car":        [0.0, 1.0, 1.0],  # cyan
    "Pedestrian": [1.0, 0.0, 1.0],  # magenta
    "Cyclist":    [1.0, 1.0, 0.0],  # yellow
}

# KITTI front-facing training range — points outside are dropped before inference.
POINT_RANGE = [0.0, -39.68, -3.0, 69.12, 39.68, 1.0]

# Raw points (x, y, z, intensity) — model expects the original Velodyne format,
# not our cleaned/ground-removed cloud.
raw = np.fromfile("0000000001.bin", dtype=np.float32).reshape(-1, 4)
rng_mask = (
    (raw[:, 0] > POINT_RANGE[0]) & (raw[:, 0] < POINT_RANGE[3])
    & (raw[:, 1] > POINT_RANGE[1]) & (raw[:, 1] < POINT_RANGE[4])
    & (raw[:, 2] > POINT_RANGE[2]) & (raw[:, 2] < POINT_RANGE[5])
)
pts_in = raw[rng_mask]
print(f"Inference input: {len(pts_in)} / {len(raw)} points after range filter")

model = PointPillars(nclasses=3)
model.load_state_dict(
    torch.load(THIRD_PARTY / "pretrained" / "epoch_160.pth",
               map_location="cpu", weights_only=True)
)
model.eval()

with torch.no_grad():
    result = model(batched_pts=[torch.from_numpy(pts_in)], mode="test")[0]

lidar_bboxes = np.asarray(result["lidar_bboxes"])  # (N, 7) = [x, y, z, dx, dy, dz, yaw]
dl_labels = np.asarray(result["labels"])
dl_scores = np.asarray(result["scores"])
print(f"Raw detections: {len(dl_scores)}")

# Top-k preview
for i in np.argsort(-dl_scores)[:5]:
    cls = LABEL2CLASS[int(dl_labels[i])]
    c = lidar_bboxes[i, :3]
    print(f"  {cls:11s} score={dl_scores[i]:.3f}  center=({c[0]:+6.2f}, {c[1]:+6.2f}, {c[2]:+6.2f})")

Inference input: 63114 / 125826 points after range filter
Raw detections: 23
  Pedestrian  score=0.865  center=( +9.13,  -5.65,  -1.46)
  Pedestrian  score=0.608  center=( +8.11,  -8.17,  -1.40)
  Car         score=0.453  center=( +8.16, +13.18,  -1.36)
  Pedestrian  score=0.257  center=( +8.49,  -8.78,  -1.48)
  Pedestrian  score=0.208  center=(+19.12, -18.31,  -1.34)


c:\Users\kaiho\Downloads\Python\point-cloud-visualisation\.venv\Lib\site-packages\torch\functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4383.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [14]:
# Build Open3D OBBs from detections; filter by score.
SCORE_THRESH = 0.2

def make_obb(center, extent, yaw, color):
    R = o3d.geometry.OrientedBoundingBox.get_rotation_matrix_from_axis_angle(
        np.array([0.0, 0.0, float(yaw)])
    )
    obb = o3d.geometry.OrientedBoundingBox(
        center=np.asarray(center, dtype=np.float64),
        R=R,
        extent=np.asarray(extent, dtype=np.float64),
    )
    obb.color = color
    return obb

obbs = []
for i in range(len(dl_scores)):
    if dl_scores[i] < SCORE_THRESH:
        continue
    cls = LABEL2CLASS[int(dl_labels[i])]
    obbs.append(make_obb(
        center=lidar_bboxes[i, :3],
        extent=lidar_bboxes[i, 3:6],
        yaw=lidar_bboxes[i, 6],
        color=CLASS_COLOR[cls],
    ))

print(f"Rendering {len(obbs)} boxes (score >= {SCORE_THRESH})")

# Greyed raw cloud as background context
raw_pcd = o3d.geometry.PointCloud()
raw_pcd.points = o3d.utility.Vector3dVector(raw[:, :3])
raw_pcd.paint_uniform_color([0.55, 0.55, 0.55])

# Small origin axes for orientation
axes = o3d.geometry.TriangleMesh.create_coordinate_frame(size=2.0, origin=[0, 0, 0])

o3d.visualization.draw_geometries([raw_pcd, axes, *obbs])

Rendering 5 boxes (score >= 0.2)


# END